# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an interactive and reproducible workflow for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
- Croissant schema: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
- Dataset DOI: [10.71728/senscience.vq4a-28xa](https://sen.science/doi/10.71728/senscience.vq4a-28xa)
- Citation: Kulka, M., Rodriguez Miera, S. and Marcet-Palacios, M. 2026 Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers.

In [ ]:
# Install mlcroissant if not already installed
!pip install -q mlcroissant


## 1. Data Loading

Load metadata and explore the dataset synopsis defined by the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview

List available record sets and their fields. Each entity is referenced by its `@id` as defined in the Croissant schema.

In [ ]:
# Display all record sets, their IDs, and field IDs
print('Record sets and field overview:')

record_set_ids = []
for recset in metadata.record_sets:
    # Each record set can be referenced via recset['@id']
    recset_id = recset['@id']
    record_set_ids.append(recset_id)
    print(f"- RecordSet @id: {recset_id}, name: {recset.get('name', '')}")
    fields = recset.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields (reference by @id):")
    for field in fields:
        field_id = field['@id'] if isinstance(field, dict) else field
        print(f"    - {field_id}")
    print()

if not record_set_ids:
    print("No record sets were found in the schema. If this occurs, check the schema or contact the data provider.")

## 3. Data Extraction

Load data from a selected record set into a pandas DataFrame. Use the RecordSet and Field `@id`s from above.

In [ ]:
# Example: extract all record sets
# You may need to update this cell if you want to analyze a specific record set
dataframes = {}
loaded_record_sets = []

for record_set_id in record_set_ids:
    try:
        print(f"Loading records from RecordSet @id: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            loaded_record_sets.append(record_set_id)
            print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
        else:
            print("(No records found in this record set.)")
    except Exception as e:
        print(f"Could not load records for {record_set_id} due to error: {e}")
    print()

# Use the first successfully loaded record set for further analysis
if loaded_record_sets:
    main_record_set_id = loaded_record_sets[0]
    print(f"Using RecordSet '{main_record_set_id}' for analysis")
    display(dataframes[main_record_set_id].head())
else:
    print("No dataframes loaded for analysis.")

## 4. Exploratory Data Analysis (EDA)

Apply data processing such as filtering or normalizing on fields, using their `@id` as references. This may involve selecting a quantitative field (e.g., peptide counts, coverage) if available.

In [ ]:
# For demonstration, automatically select the first numeric field from DataFrame
import numpy as np

if loaded_record_sets:
    df = dataframes[main_record_set_id]
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        # Try to infer numerics from object columns
        object_cols = df.select_dtypes(include=[object]).columns
        for col in object_cols:
            try:
                converted = pd.to_numeric(df[col], errors='coerce')
                if converted.notna().sum() > 0:
                    df[col] = converted
                    numeric_cols.append(col)
            except:
                continue

    if numeric_cols:
        numeric_field_id = numeric_cols[0] # Use its name as id
        print(f"Selected numeric field: {numeric_field_id}")

        threshold = df[numeric_field_id].quantile(0.75)  # 75th percentile as example
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (top 25%): {len(filtered_df)} rows")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical field (first non-numeric field not an index)
        non_numeric = [col for col in df.columns if col not in numeric_cols and not pd.api.types.is_datetime64_any_dtype(df[col])]
        if non_numeric:
            group_field = non_numeric[0]
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable non-numeric group field found.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No data loaded for EDA.")

## 5. Visualization

Visualize the distribution of the selected numeric field and any relationships (e.g., group means) if fields are available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if loaded_record_sets and numeric_cols:
    # Histogram of main numeric field (all values)
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot, optionally grouped by group_field if available
    if 'group_field' in locals():
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric columns available for visualization.")

## 6. Conclusion

- In this notebook, we demonstrated how to use the FAIR² Croissant schema and the `mlcroissant` library to:
  - Load and inspect dataset metadata and structure by `@id`.
  - Extract and explore tabular data from all available record sets.
  - Apply basic exploratory analysis and visualization to a quantitative field.
  
Refer to the [FAIR² documentation](https://mlcommons.org/croissant/) for more details and advanced data handling.